In [1]:
#| label: laser-qa-summary-functions
#| output: false


def build_laser_qa_summary(
    maximum_power,
    threshold_summary,
):
    """Build one operational QA row per wavelength."""

    columns = [
        "wavelength_nm",
        "latest_month",
        "latest_power_mW",
        "previous_power_mW",
        "change_percent",
        "reference_month",
        "reference_maximum_mW",
        "out_of_spec_threshold_mW",
        "percent_of_reference",
        "measurement_count",
        "status",
    ]

    if maximum_power.empty:
        return pd.DataFrame(columns=columns)

    thresholds = {}

    if not threshold_summary.empty:
        thresholds = (
            threshold_summary
            .set_index("wavelength_nm")
            .to_dict(orient="index")
        )

    rows = []

    for wavelength, group in maximum_power.groupby(
        "wavelength", sort=True
    ):
        group = group.sort_values("date").reset_index(drop=True)
        latest = group.iloc[-1]
        previous = group.iloc[-2] if len(group) > 1 else None
        latest_power = float(latest["power_maximum"])
        previous_power = (
            float(previous["power_maximum"])
            if previous is not None
            else None
        )
        threshold = thresholds.get(int(wavelength), {})
        reference_power = threshold.get("reference_maximum_mW")
        threshold_power = threshold.get("out_of_spec_threshold_mW")

        change_percent = None
        if previous_power not in (None, 0):
            change_percent = (
                (latest_power / previous_power) - 1
            ) * 100

        percent_of_reference = None
        if reference_power not in (None, 0):
            percent_of_reference = (
                latest_power / float(reference_power)
            ) * 100

        if threshold_power is None:
            status = "No baseline"
        elif latest_power < float(threshold_power):
            status = "Review"
        else:
            status = "Within spec"

        rows.append(
            {
                "wavelength_nm": int(wavelength),
                "latest_month": latest["month"],
                "latest_power_mW": latest_power,
                "previous_power_mW": previous_power,
                "change_percent": change_percent,
                "reference_month": threshold.get("reference_month"),
                "reference_maximum_mW": reference_power,
                "out_of_spec_threshold_mW": threshold_power,
                "percent_of_reference": percent_of_reference,
                "measurement_count": len(group),
                "status": status,
            }
        )

    return pd.DataFrame(rows, columns=columns)


In [2]:
#| label: setup
#| output: false

from __future__ import annotations

import io
import os
import re

from contextlib import redirect_stdout
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go


# --------------------------------------------------
# Project directories
# --------------------------------------------------

PROJECT_DIR = Path(
    os.environ.get(
        "QUARTO_PROJECT_DIR",
        ".",
    )
).resolve()


DATA_DIR = (
    PROJECT_DIR
    / "data"
)

LASER_POWER_DIR = (
    DATA_DIR
    / "Laser_Power_Measurements"
)

OUTPUT_ROOT = (
    PROJECT_DIR
    / "outputs"
)


DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------
# Detect microscope directories
# --------------------------------------------------

if LASER_POWER_DIR.exists():

    MICROSCOPE_DIRS = sorted(
        [
            folder
            for folder
            in LASER_POWER_DIR.iterdir()
            if folder.is_dir()
            and not folder.name.startswith(".")
        ],
        key=lambda folder: folder.name.lower(),
    )

else:

    MICROSCOPE_DIRS = []


print(
    "Detected Laser Power microscopes:",
    ", ".join(
        folder.name
        for folder
        in MICROSCOPE_DIRS
    )
    or "None",
)

Detected Laser Power microscopes: Andor_Dragonfly_620, Leica_SP8_AOBS, Leica_SP8_Icahn, Leica_STED, Leica_Stellaris_Atran, Leica_Stellaris_W619, Zeiss_LSM780, Zeiss_LSM880, Zeiss_LSM980


In [3]:
#| label: parsing-functions
#| output: false


# --------------------------------------------------
# Expected filename format
# --------------------------------------------------

FILENAME_PATTERN = re.compile(
    (
        r"^(?P<month>\d{2})-"
        r"(?P<year>\d{2}|\d{4})_"
        r"(?P<wavelength>\d+)\.csv$"
    ),
    flags=re.IGNORECASE,
)


# --------------------------------------------------
# Parse filename
# --------------------------------------------------

def parse_measurement_filename(
    file_path: Path,
) -> dict[str, Any]:
    """
    Parse filenames such as:

        02-26_405.csv

    Returns:
        wavelength
        date
        month
        legend label
    """

    match = FILENAME_PATTERN.match(
        file_path.name
    )

    if not match:

        raise ValueError(
            f"Unexpected filename "
            f"'{file_path.name}'. "
            f"Expected MM-YY_WAVELENGTH.csv."
        )


    month = int(
        match.group("month")
    )

    year_text = (
        match.group("year")
    )

    year = int(
        year_text
    )


    # Convert 2-digit years
    if len(year_text) == 2:

        year += 2000


    try:

        measurement_date = pd.Timestamp(
            year=year,
            month=month,
            day=1,
        )

    except ValueError as exc:

        raise ValueError(
            f"Invalid month/year in "
            f"'{file_path.name}'."
        ) from exc


    return {
        "path": file_path,
        "wavelength": (
            match.group("wavelength")
        ),
        "date": measurement_date,
        "month": (
            measurement_date.strftime(
                "%Y-%m"
            )
        ),
        "legend_label": (
            measurement_date.strftime(
                "%b %Y"
            )
        ),
    }


# --------------------------------------------------
# Read a section from CSV
# --------------------------------------------------

def read_semicolon_section(
    file_path: Path,
    section_names: tuple[str, ...],
) -> pd.DataFrame:
    """
    Read a semicolon-delimited section from
    the microscope laser-power CSV export.
    """

    lines = file_path.read_text(
        encoding="utf-8-sig",
        errors="replace",
    ).splitlines()


    section_names_lower = tuple(
        name.lower()
        for name in section_names
    )


    # ----------------------------------------------
    # Locate section marker
    # ----------------------------------------------

    marker_index = None


    for index, line in enumerate(lines):

        line_lower = (
            line.strip().lower()
        )

        if any(
            name in line_lower
            for name
            in section_names_lower
        ):

            marker_index = index
            break


    if marker_index is None:

        expected = " or ".join(
            repr(name)
            for name
            in section_names
        )

        raise ValueError(
            f"No {expected} section found "
            f"in {file_path.name}."
        )


    # ----------------------------------------------
    # Locate section header
    # ----------------------------------------------

    header_index = None


    for index in range(
        marker_index + 1,
        len(lines),
    ):

        if lines[index].strip():

            header_index = index
            break


    if header_index is None:

        raise ValueError(
            f"No header found after "
            f"the section marker in "
            f"{file_path.name}."
        )


    header = [
        value.strip()
        for value
        in lines[header_index].split(";")
    ]


    rows: list[list[str]] = []


    # ----------------------------------------------
    # Read section rows
    # ----------------------------------------------

    for line in lines[
        header_index + 1:
    ]:

        stripped = (
            line.strip()
        )


        if not stripped:

            break


        if stripped.lower().startswith(
            "time"
        ):

            break


        values = [
            value.strip()
            for value
            in stripped.split(";")
        ]


        if len(values) < len(header):

            values.extend(
                [""] * (
                    len(header)
                    - len(values)
                )
            )


        elif len(values) > len(header):

            values = values[
                :len(header)
            ]


        rows.append(
            values
        )


    if not rows:

        raise ValueError(
            f"The requested section in "
            f"{file_path.name} "
            f"contains no data rows."
        )


    return pd.DataFrame(
        rows,
        columns=header,
    )


# --------------------------------------------------
# Convert numeric columns
# --------------------------------------------------

def convert_numeric_columns(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    """
    Convert columns to numeric where all
    nonblank values can be converted.
    """

    result = (
        dataframe.copy()
    )


    for column in result.columns:

        text = (
            result[column]
            .astype(str)
            .str.strip()
        )


        numeric = pd.to_numeric(
            text.str.replace(
                ",",
                ".",
                regex=False,
            ),
            errors="coerce",
        )


        nonblank = (
            text.ne("")
        )


        if (
            nonblank.any()
            and numeric[
                nonblank
            ].notna().all()
        ):

            result[column] = numeric


    return result


# --------------------------------------------------
# Read laser calibration table
# --------------------------------------------------

def read_power_instruction_table(
    file_path: Path,
) -> pd.DataFrame:
    """
    Read the Result table values section used
    to build laser calibration curves.
    """

    dataframe = read_semicolon_section(
        file_path,
        (
            "Result table values",
        ),
    )


    dataframe.columns = [
        column.strip()
        for column
        in dataframe.columns
    ]


    dataframe = dataframe.rename(
        columns={
            "power_instruction":
                "power_percentage_values"
        }
    )


    if (
        "power_percentage_values"
        not in dataframe.columns
    ):

        raise ValueError(
            "The Result table values section "
            f"in {file_path.name} does not "
            "contain a power_instruction column."
        )


    return convert_numeric_columns(
        dataframe
    )


# --------------------------------------------------
# Read maximum laser power
# --------------------------------------------------

def read_maximum_power(
    file_path: Path,
) -> float | None:
    """
    Read the maximum-power value from either:

        Result values

    or:

        Primary metrics values
    """

    try:

        dataframe = (
            read_semicolon_section(
                file_path,
                (
                    "Result values",
                    "Primary metrics values",
                ),
            )
        )

    except ValueError as exc:

        print(
            f"Warning: {exc}"
        )

        return None


    dataframe.columns = [
        column.strip()
        for column
        in dataframe.columns
    ]


    # ----------------------------------------------
    # Prefer clearly named maximum-power column
    # ----------------------------------------------

    named_candidates = [
        column
        for column
        in dataframe.columns
        if (
            "power"
            in column.lower()
        )
        and (
            "max"
            in column.lower()
            or "maximum"
            in column.lower()
        )
    ]


    candidate_series = []


    if named_candidates:

        candidate_series.append(
            dataframe[
                named_candidates[0]
            ]
        )


    # Original notebook fallback:
    # use third column
    if dataframe.shape[1] >= 3:

        candidate_series.append(
            dataframe.iloc[:, 2]
        )


    # ----------------------------------------------
    # Find first numeric maximum
    # ----------------------------------------------

    for series in candidate_series:

        numeric = pd.to_numeric(
            series
            .astype(str)
            .str.strip()
            .str.replace(
                ",",
                ".",
                regex=False,
            ),
            errors="coerce",
        ).dropna()


        if not numeric.empty:

            return float(
                numeric.iloc[0]
            )


    print(
        "Warning: No numeric maximum-power "
        f"value found in {file_path.name}."
    )

    return None

In [4]:
#| label: calibration-functions
#| output: false


# --------------------------------------------------
# Combine monthly calibration measurements
# --------------------------------------------------

def combine_calibration_tables(
    records: list[
        tuple[
            dict[str, Any],
            pd.DataFrame,
        ]
    ],
) -> pd.DataFrame:
    """
    Merge monthly calibration measurements
    for one wavelength.
    """

    combined: pd.DataFrame | None = None

    seen_months: set[str] = set()


    for metadata, dataframe in sorted(
        records,
        key=lambda item: item[0]["date"],
    ):

        month = (
            metadata["month"]
        )


        if month in seen_months:

            raise ValueError(
                "More than one calibration "
                f"file was found for "
                f"{metadata['wavelength']} nm "
                f"in {month}."
            )


        seen_months.add(
            month
        )


        other_columns = [
            column
            for column
            in dataframe.columns
            if (
                column
                != "power_percentage_values"
            )
        ]


        renamed = (
            dataframe
            .rename(
                columns={
                    column:
                    f"{month}_{column}"
                    for column
                    in other_columns
                }
            )
            .drop_duplicates(
                subset=[
                    "power_percentage_values"
                ]
            )
        )


        if combined is None:

            combined = renamed

        else:

            combined = pd.merge(
                combined,
                renamed,
                on="power_percentage_values",
                how="outer",
                validate="one_to_one",
            )


    if combined is None:

        return pd.DataFrame()


    return (
        combined
        .drop_duplicates(
            subset=[
                "power_percentage_values"
            ]
        )
        .sort_values(
            "power_percentage_values"
        )
        .reset_index(
            drop=True
        )
    )


# --------------------------------------------------
# Plot calibration curves
# --------------------------------------------------

def plot_calibration_curves(
    combined_all: dict[str, pd.DataFrame],
    save_folder: Path,
) -> list[Path]:
    """
    Create one interactive Plotly laser-power
    calibration plot per wavelength.
    """

    save_folder.mkdir(
        parents=True,
        exist_ok=True,
    )

    generated: list[Path] = []

    for wavelength in sorted(
        combined_all,
        key=int,
    ):

        dataframe = combined_all[
            wavelength
        ]

        # ------------------------------------------
        # Detect available measurement months
        # ------------------------------------------

        extracted_months = set()

        for column in dataframe.columns:

            if f"_{wavelength}" in column:

                match = re.match(
                    r"^(\d{4}-\d{2})",
                    column,
                )

                if match:

                    extracted_months.add(
                        match.group(1)
                    )

        month_keys = sorted(
            extracted_months,
            key=lambda month:
                pd.Timestamp(
                    f"{month}-01"
                ),
        )

        if not month_keys:

            print(
                f"Warning: No calibration "
                f"columns found for "
                f"{wavelength} nm."
            )

            continue

        # ------------------------------------------
        # Create Plotly figure
        # ------------------------------------------

        figure = go.Figure()

        plotted_series = 0

        for month in month_keys:

            # --------------------------------------
            # Find power column
            # --------------------------------------

            power_columns = [
                column
                for column
                in dataframe.columns
                if (
                    column.startswith(month)
                    and f"_{wavelength}"
                    in column
                    and "error"
                    not in column.lower()
                )
            ]

            if not power_columns:

                continue

            power_column = (
                power_columns[0]
            )

            # --------------------------------------
            # Find error column
            # --------------------------------------

            error_columns = [
                column
                for column
                in dataframe.columns
                if (
                    column.startswith(month)
                    and "error"
                    in column.lower()
                )
            ]

            plot_data = pd.DataFrame(
                {
                    "PowerPercentage":
                        pd.to_numeric(
                            dataframe[
                                "power_percentage_values"
                            ],
                            errors="coerce",
                        ),

                    "Power":
                        pd.to_numeric(
                            dataframe[
                                power_column
                            ],
                            errors="coerce",
                        ),
                }
            )

            if error_columns:

                plot_data["Error"] = (
                    pd.to_numeric(
                        dataframe[
                            error_columns[0]
                        ],
                        errors="coerce",
                    )
                    .abs()
                )

            plot_data = (
                plot_data
                .dropna(
                    subset=[
                        "PowerPercentage",
                        "Power",
                    ]
                )
                .sort_values(
                    "PowerPercentage"
                )
            )

            if plot_data.empty:

                continue

            # --------------------------------------
            # Error bars
            # --------------------------------------

            error_y = None

            if "Error" in plot_data.columns:

                error_y = dict(
                    type="data",
                    array=plot_data[
                        "Error"
                    ],
                    visible=True,
                )

            # --------------------------------------
            # Add trace
            # --------------------------------------

            figure.add_trace(
                go.Scatter(
                    x=plot_data[
                        "PowerPercentage"
                    ],
                    y=plot_data[
                        "Power"
                    ],
                    mode="lines+markers",
                    name=(
                        pd.Timestamp(
                            f"{month}-01"
                        )
                        .strftime(
                            "%b %Y"
                        )
                    ),
                    line=dict(
                        width=2.5,
                    ),
                    marker=dict(
                        size=7,
                        line=dict(width=1, color="white"),
                    ),
                    error_y=error_y,
                    hovertemplate=(
                        "<b>%{fullData.name}</b>"
                        "<br>"
                        "Laser setting: "
                        "%{x:.1f}%"
                        "<br>"
                        "Measured power: "
                        "%{y:.3f} µW"
                        "<extra></extra>"
                    ),
                )
            )

            plotted_series += 1

        if plotted_series == 0:

            continue

        # ------------------------------------------
        # Layout
        # ------------------------------------------

        figure.update_layout(
            title=(
                "Laser Power Calibration - "
                f"{wavelength} nm"
            ),
            xaxis_title=(
                "Power Percentage Value"
            ),
            yaxis_title=(
                "Measured Power (µW)"
            ),
            template="plotly_white",
            hovermode="closest",
            autosize=True,
            paper_bgcolor="#ffffff",
            plot_bgcolor="#ffffff",
            font=dict(
                family="Arial, sans-serif",
                color="#334155",
            ),
            title_font=dict(color="#1a2e5a", size=18),

            legend=dict(
                title="Measurement Month",
                orientation="h",
                yanchor="bottom",
                y=1.02,
                xanchor="right",
                x=1,
            ),

            margin=dict(
                l=70,
                r=30,
                t=110,
                b=70,
            ),
        )

        figure.update_xaxes(
            showgrid=True,
            gridcolor="#e8edf3",
            zeroline=False,
        )

        figure.update_yaxes(
            showgrid=True,
            gridcolor="#e8edf3",
            zeroline=False,
        )

        # ------------------------------------------
        # Save interactive HTML
        # ------------------------------------------

        plot_path = (
            save_folder
            / (
                f"laser_power_"
                f"{wavelength}nm.html"
            )
        )

        figure.write_html(
            plot_path,
            include_plotlyjs="cdn",
            full_html=True,
            config={
                "responsive": True,
                "displaylogo": False,
                "scrollZoom": False,
            },
        )

        generated.append(
            plot_path
        )

        print(
            "Saved interactive calibration plot: "
            f"{plot_path}"
        )

    return generated


In [5]:
#| label: maximum-power-functions
#| output: false


def plot_maximum_power_trends(
    maximum_power: pd.DataFrame,
    save_folder: Path,
    target_month: str | None,
) -> tuple[list[Path], pd.DataFrame]:
    """
    Create interactive maximum laser-power plots.

    Out-of-spec threshold:
        70% of reference maximum.

    If target_month is unavailable for a
    wavelength, use the latest available month.
    """

    generated: list[Path] = []

    threshold_rows: list[
        dict[str, Any]
    ] = []

    if maximum_power.empty:

        return (
            generated,
            pd.DataFrame(),
        )

    save_folder.mkdir(
        parents=True,
        exist_ok=True,
    )

    # ----------------------------------------------
    # Process each wavelength
    # ----------------------------------------------

    for wavelength, group in (
        maximum_power.groupby(
            "wavelength",
            sort=True,
        )
    ):

        group = (
            group
            .sort_values(
                "date"
            )
            .reset_index(
                drop=True
            )
        )

        group[
            "power_maximum"
        ] = pd.to_numeric(
            group[
                "power_maximum"
            ],
            errors="coerce",
        )

        group = group.dropna(
            subset=[
                "date",
                "power_maximum",
            ]
        )

        if group.empty:

            continue

        # ------------------------------------------
        # Determine reference month
        # ------------------------------------------

        available_months = (
            group["month"]
            .astype(str)
            .tolist()
        )

        latest_month = (
            group.iloc[-1][
                "month"
            ]
        )

        if (
            target_month is not None
            and target_month
            in available_months
        ):

            reference_month = (
                target_month
            )

        else:

            reference_month = (
                latest_month
            )

            if target_month is not None:

                print(
                    f"Warning: "
                    f"{target_month} is not "
                    f"available for "
                    f"{wavelength} nm. "
                    f"Using {reference_month}."
                )

        # ------------------------------------------
        # Reference maximum
        # ------------------------------------------

        reference_rows = (
            group.loc[
                group[
                    "month"
                ]
                == reference_month
            ]
        )

        reference_maximum = float(
            reference_rows.iloc[-1][
                "power_maximum"
            ]
        )

        # ------------------------------------------
        # 70% threshold
        # ------------------------------------------

        threshold_value = (
            reference_maximum
            * 0.70
        )

        threshold_rows.append(
            {
                "wavelength_nm":
                    int(wavelength),

                "reference_month":
                    reference_month,

                "reference_maximum_mW":
                    reference_maximum,

                "out_of_spec_threshold_mW":
                    threshold_value,
            }
        )

        # ------------------------------------------
        # Status
        # ------------------------------------------

        group["Status"] = (
            group[
                "power_maximum"
            ].apply(
                lambda value:
                    (
                        "Out of Spec"
                        if value
                        < threshold_value
                        else "Within Spec"
                    )
            )
        )

        # ------------------------------------------
        # Create Plotly figure
        # ------------------------------------------

        figure = go.Figure()

        # Main measurement trace
        figure.add_trace(
            go.Scatter(
                x=group[
                    "date"
                ],
                y=group[
                    "power_maximum"
                ],
                mode="lines+markers",
                name="Maximum Laser Power",

                line=dict(
                    color="#0787b5",
                    width=2.5,
                ),

                marker=dict(
                    color="#0787b5",
                    size=9,
                    line=dict(width=1, color="white"),
                ),

                customdata=group[
                    [
                        "Status",
                        "month",
                    ]
                ],

                hovertemplate=(
                    "<b>Maximum Laser Power</b>"
                    "<br>"
                    "Date: %{x|%b %Y}"
                    "<br>"
                    "Power: %{y:.3f} mW"
                    "<br>"
                    "Status: "
                    "%{customdata[0]}"
                    "<extra></extra>"
                ),
            )
        )

        # ------------------------------------------
        # Highlight failing measurements
        # ------------------------------------------

        out_of_spec = group[
            group[
                "power_maximum"
            ]
            < threshold_value
        ]

        if not out_of_spec.empty:

            figure.add_trace(
                go.Scatter(
                    x=out_of_spec[
                        "date"
                    ],
                    y=out_of_spec[
                        "power_maximum"
                    ],
                    mode="markers",
                    name=(
                        "Out-of-Spec Measurement"
                    ),
                    marker=dict(
                        color="#c2410c",
                        size=12,
                    ),
                    hovertemplate=(
                        "<b>Out of Spec</b>"
                        "<br>"
                        "Date: %{x|%b %Y}"
                        "<br>"
                        "Power: %{y:.3f} mW"
                        "<extra></extra>"
                    ),
                )
            )

        # ------------------------------------------
        # Reference line
        # ------------------------------------------

        figure.add_hline(
            y=reference_maximum,
            line_dash="dot",
            line_color="gray",
            line_width=1.5,

            annotation_text=(
                f"Reference: "
                f"{reference_maximum:.2f} mW"
            ),

            annotation_position=(
                "top left"
            ),
        )

        # ------------------------------------------
        # Out-of-spec threshold
        # ------------------------------------------

        figure.add_hline(
            y=threshold_value,
            line_dash="dash",
            line_color="#c2410c",
            line_width=2,

            annotation_text=(
                "Out-of-spec threshold "
                f"(70%): "
                f"{threshold_value:.2f} mW"
            ),

            annotation_position=(
                "bottom left"
            ),
        )

        # ------------------------------------------
        # Layout
        # ------------------------------------------

        figure.update_layout(
            title=(
                "Maximum Laser Power - "
                f"{wavelength} nm"
            ),

            xaxis_title=(
                "Measurement Month"
            ),

            yaxis_title=(
                "Measured Maximum Power (mW)"
            ),

            template="plotly_white",

            hovermode="closest",

            autosize=True,
            paper_bgcolor="#ffffff",
            plot_bgcolor="#ffffff",
            font=dict(
                family="Arial, sans-serif",
                color="#334155",
            ),
            title_font=dict(color="#1a2e5a", size=18),

            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=1.02,
                xanchor="right",
                x=1,
            ),

            margin=dict(
                l=70,
                r=30,
                t=110,
                b=70,
            ),
        )

        figure.update_xaxes(
            tickformat="%b %Y",
            showgrid=True,
            gridcolor="#e8edf3",
            zeroline=False,
        )

        figure.update_yaxes(
            showgrid=True,
            gridcolor="#e8edf3",
            zeroline=False,
        )

        # ------------------------------------------
        # Save HTML
        # ------------------------------------------

        plot_path = (
            save_folder
            / (
                "laser_power_max_"
                f"{wavelength}nm.html"
            )
        )

        figure.write_html(
            plot_path,
            include_plotlyjs="cdn",
            full_html=True,
            config={
                "responsive": True,
                "displaylogo": False,
                "scrollZoom": False,
            },
        )

        generated.append(
            plot_path
        )

        print(
            "Saved interactive "
            "maximum-power plot: "
            f"{plot_path}"
        )

    # ----------------------------------------------
    # Threshold summary
    # ----------------------------------------------

    threshold_summary = (
        pd.DataFrame(
            threshold_rows
        )
    )

    if not threshold_summary.empty:

        threshold_summary = (
            threshold_summary
            .sort_values(
                "wavelength_nm"
            )
            .reset_index(
                drop=True
            )
        )

    return (
        generated,
        threshold_summary,
    )


In [6]:
#| label: excel-functions
#| output: false


def write_excel_workbook(
    combined_all: dict[
        str,
        pd.DataFrame,
    ],
    maximum_power: pd.DataFrame,
    threshold_summary: pd.DataFrame,
    output_path: Path,
) -> None:
    """
    Write calibration measurements,
    maximum-power measurements,
    and QA thresholds to Excel.
    """

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    # --------------------------------------------------
    # Write sheets
    # --------------------------------------------------

    with pd.ExcelWriter(
        output_path,
        engine="openpyxl",
    ) as writer:

        # Calibration sheets
        for wavelength in sorted(
            combined_all,
            key=int,
        ):

            combined_all[
                wavelength
            ].to_excel(
                writer,
                sheet_name=(
                    f"{wavelength}nm"
                ),
                index=False,
            )


        # Maximum-power sheet
        if not maximum_power.empty:

            maximum_power.loc[
                :,
                [
                    "month",
                    "wavelength",
                    "power_maximum",
                    "source_file",
                ],
            ].to_excel(
                writer,
                sheet_name="Maximum Power",
                index=False,
            )


        # Threshold summary
        if not threshold_summary.empty:

            threshold_summary.to_excel(
                writer,
                sheet_name=(
                    "Threshold Summary"
                ),
                index=False,
            )


    # --------------------------------------------------
    # Format workbook
    # --------------------------------------------------

    from openpyxl import (
        load_workbook,
    )


    workbook = load_workbook(
        output_path
    )


    for worksheet in (
        workbook.worksheets
    ):

        worksheet.freeze_panes = (
            "A2"
        )


        worksheet.auto_filter.ref = (
            worksheet.dimensions
        )


        for column_cells in (
            worksheet.columns
        ):

            values = [
                str(cell.value)
                if cell.value
                is not None
                else ""
                for cell
                in column_cells
            ]


            width = min(
                max(
                    max(
                        (
                            len(value)
                            for value
                            in values
                        ),
                        default=0,
                    )
                    + 2,
                    12,
                ),
                36,
            )


            worksheet.column_dimensions[
                column_cells[
                    0
                ].column_letter
            ].width = width


    workbook.save(
        output_path
    )

In [7]:
#| label: target-month-functions
#| output: false


def get_target_month(
    microscope_dir: Path,
) -> str | None:
    """
    Read an optional microscope-specific
    target month.

    Expected file:

        target_month.txt

    Expected content:

        YYYY-MM

    Example:

        2026-02
    """

    target_file = (
        microscope_dir
        / "target_month.txt"
    )


    if not target_file.exists():

        return None


    target_month = (
        target_file
        .read_text(
            encoding="utf-8"
        )
        .strip()
    )


    if not target_month:

        return None


    try:

        pd.Period(
            target_month,
            freq="M",
        )

    except ValueError as exc:

        raise ValueError(
            f"{microscope_dir.name}/"
            f"target_month.txt must contain "
            f"a month in YYYY-MM format. "
            f"Found: {target_month}"
        ) from exc


    return target_month

In [8]:
#| label: workflow-functions
#| output: false


def run_analysis(
    input_folder: Path,
    output_excel: Path,
    plot_folder: Path,
    target_month: str | None = None,
) -> dict[str, Any]:
    """
    Run complete laser-power QA analysis
    for one microscope.
    """

    # --------------------------------------------------
    # Validate target month
    # --------------------------------------------------

    if target_month is not None:

        try:

            pd.Period(
                target_month,
                freq="M",
            )

        except ValueError as exc:

            raise ValueError(
                "Target month must use "
                "YYYY-MM format."
            ) from exc


    # --------------------------------------------------
    # Find CSV files
    # --------------------------------------------------

    candidate_files = sorted(
        input_folder.glob(
            "*.csv"
        )
    )


    parsed_files: list[
        dict[str, Any]
    ] = []


    ignored_files: list[str] = []


    for file_path in candidate_files:

        try:

            parsed_files.append(
                parse_measurement_filename(
                    file_path
                )
            )

        except ValueError as exc:

            ignored_files.append(
                file_path.name
            )

            print(
                f"Warning: {exc}"
            )


    # --------------------------------------------------
    # No usable data
    # --------------------------------------------------

    if not parsed_files:

        return {
            "status": "no_data",

            "message": (
                "No valid laser-power CSV "
                f"files were found in "
                f"{input_folder}."
            ),

            "ignored_files":
                ignored_files,

            "plots": [],
        }


    # --------------------------------------------------
    # Collect measurements
    # --------------------------------------------------

    instruction_records: dict[
        str,
        list[
            tuple[
                dict[str, Any],
                pd.DataFrame,
            ]
        ],
    ] = {}


    maximum_rows: list[
        dict[str, Any]
    ] = []


    skipped_instruction_files: (
        list[str]
    ) = []


    for metadata in parsed_files:

        file_path = (
            metadata["path"]
        )


        # ------------------------------------------
        # Calibration curve
        # ------------------------------------------

        try:

            instruction_table = (
                read_power_instruction_table(
                    file_path
                )
            )


            instruction_records.setdefault(
                metadata[
                    "wavelength"
                ],
                [],
            ).append(
                (
                    metadata,
                    instruction_table,
                )
            )


        except ValueError as exc:

            skipped_instruction_files.append(
                file_path.name
            )


            print(
                "Warning: Calibration "
                f"analysis skipped for "
                f"{file_path.name}: "
                f"{exc}"
            )


        # ------------------------------------------
        # Maximum power
        # ------------------------------------------

        maximum_value = (
            read_maximum_power(
                file_path
            )
        )


        if maximum_value is not None:

            maximum_rows.append(
                {
                    "date":
                        metadata[
                            "date"
                        ],

                    "month":
                        metadata[
                            "month"
                        ],

                    "wavelength":
                        int(
                            metadata[
                                "wavelength"
                            ]
                        ),

                    "power_maximum":
                        maximum_value,

                    "source_file":
                        file_path.name,
                }
            )


    # --------------------------------------------------
    # Combine calibration measurements
    # --------------------------------------------------

    combined_all = {
        wavelength:
            combine_calibration_tables(
                records
            )

        for wavelength, records
        in instruction_records.items()

        if records
    }


    # --------------------------------------------------
    # Maximum-power dataframe
    # --------------------------------------------------

    maximum_power = pd.DataFrame(
        maximum_rows
    )


    if not maximum_power.empty:

        maximum_power[
            "date"
        ] = pd.to_datetime(
            maximum_power[
                "date"
            ]
        )


        maximum_power = (
            maximum_power
            .sort_values(
                [
                    "wavelength",
                    "date",
                    "source_file",
                ]
            )
            .reset_index(
                drop=True
            )
        )


    # --------------------------------------------------
    # Generate plots
    # --------------------------------------------------

    calibration_plots = (
        plot_calibration_curves(
            combined_all,
            plot_folder,
        )
    )


    (
        maximum_plots,
        threshold_summary,
    ) = plot_maximum_power_trends(
        maximum_power,
        plot_folder,
        target_month,
    )


    # --------------------------------------------------
    # Operational QA summary
    # --------------------------------------------------

    qa_summary = build_laser_qa_summary(
        maximum_power,
        threshold_summary,
    )

    summary_csv = output_excel.with_name(
        "laser_power_summary.csv"
    )

    if not qa_summary.empty:
        qa_summary.to_csv(summary_csv, index=False)


    # --------------------------------------------------
    # Excel workbook
    # --------------------------------------------------

    if (
        combined_all
        or not maximum_power.empty
        or not threshold_summary.empty
    ):

        write_excel_workbook(
            combined_all,
            maximum_power,
            threshold_summary,
            output_excel,
        )


        print(
            f"Saved Excel workbook: "
            f"{output_excel}"
        )


    # --------------------------------------------------
    # File summary
    # --------------------------------------------------

    file_summary = pd.DataFrame(
        [
            {
                "source_file":
                    metadata[
                        "path"
                    ].name,

                "month":
                    metadata[
                        "month"
                    ],

                "wavelength_nm":
                    int(
                        metadata[
                            "wavelength"
                        ]
                    ),
            }

            for metadata
            in parsed_files
        ]
    )


    if not file_summary.empty:

        file_summary = (
            file_summary
            .sort_values(
                [
                    "month",
                    "wavelength_nm",
                ]
            )
            .reset_index(
                drop=True
            )
        )


    # --------------------------------------------------
    # Results
    # --------------------------------------------------

    return {
        "status":
            "ok",

        "message":
            (
                f"Processed "
                f"{len(parsed_files)} "
                f"valid CSV files."
            ),

        "file_summary":
            file_summary,

        "combined_all":
            combined_all,

        "maximum_power":
            maximum_power,

        "threshold_summary":
            threshold_summary,

        "qa_summary":
            qa_summary,

        "summary_csv":
            summary_csv,

        "ignored_files":
            ignored_files,

        "skipped_instruction_files":
            skipped_instruction_files,

        "plots":
            calibration_plots
            + maximum_plots,

        "excel_path":
            output_excel,
    }

In [9]:
#| label: run-analysis
#| echo: false
#| output: false


all_results = {}


for microscope_dir in (
    MICROSCOPE_DIRS
):

    microscope = (
        microscope_dir.name
    )


    # ----------------------------------------------
    # Microscope output directories
    # ----------------------------------------------

    output_dir = (
        OUTPUT_ROOT
        / microscope
    )


    plot_dir = (
        output_dir
        / "plots"
    )


    excel_path = (
        output_dir
        / "combined_power_data.xlsx"
    )


    summary_csv = (
        output_dir
        / "laser_power_summary.csv"
    )


    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    plot_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    # ----------------------------------------------
    # Delete old generated plots
    # ----------------------------------------------

    for pattern in (
        "*.png",
        "*.html",
    ):

        for old_plot in plot_dir.glob(
            pattern
        ):

            old_plot.unlink()


    # ----------------------------------------------
    # Delete old Excel workbook
    # ----------------------------------------------

    if excel_path.exists():

        excel_path.unlink()

    if summary_csv.exists():

        summary_csv.unlink()


    # ----------------------------------------------
    # Read microscope-specific target month
    # ----------------------------------------------

    microscope_target_month = (
        get_target_month(
            microscope_dir
        )
    )


    # ----------------------------------------------
    # Run analysis
    # ----------------------------------------------

    with redirect_stdout(
        io.StringIO()
    ):

        results = run_analysis(
            input_folder=(
                microscope_dir
            ),
            output_excel=(
                excel_path
            ),
            plot_folder=(
                plot_dir
            ),
            target_month=(
                microscope_target_month
            ),
        )


    # ----------------------------------------------
    # Store metadata
    # ----------------------------------------------

    results[
        "microscope"
    ] = microscope


    results[
        "target_month"
    ] = (
        microscope_target_month
    )


    results[
        "output_dir"
    ] = output_dir


    results[
        "excel_path"
    ] = excel_path


    all_results[
        microscope
    ] = results